# 17 — Hierarchical Relationship Graph + PRIDE API + OLS

**Core idea from Gemini:** Instead of extracting each SDRF column independently,
build a biological provenance graph for each PXD first:

```
Organism (root)
  └── Tissue/Biofluid
        └── CellLine (derived from tissue)
              └── Treatment/Condition
```

Then assign SDRF columns by reading the correct level of the graph.
This prevents 'HeLa' landing in Organism, 'blood serum' contaminating a cell line study,
or 'fetal' being assigned to adult patient samples.

**The validation step:** Before writing any value, check it's consistent with
the established biological context. A CellLine node means MaterialType='cell line'.
A Tissue node without CellLine means MaterialType='tissue' or 'biofluid'.
A Disease node only fires if there's a clinical context signal.

## 0. Imports

In [1]:
import os, re, json, time, difflib
from collections import defaultdict, Counter
from pathlib import Path
from functools import lru_cache

import networkx as nx
import requests
import pandas as pd
from tqdm import tqdm

# Install networkx if needed
try:
    import networkx
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'networkx'])
    import networkx as nx

IS_KAGGLE = Path('/kaggle').exists()
if IS_KAGGLE:
    BASE_PATH = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    OUT_PATH  = Path('/kaggle/working/submission_graph.csv')
else:
    BASE_PATH = Path.cwd().parent / 'data'
    OUT_PATH  = Path.cwd().parent / 'outputs' / 'submission_graph.csv'

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TRAIN_SDRF_DIR = BASE_PATH / 'TrainingSDRFs'
SAMPLE_SUB     = BASE_PATH / 'SampleSubmission.csv'
PRIDE_TIMEOUT  = 15
PX_TIMEOUT     = 12

_pubtext_candidates = [
    BASE_PATH / 'Test_PubText' / 'Test PubText',
    BASE_PATH / 'Test_PubText',
    BASE_PATH / 'TestPubText',
    BASE_PATH / 'Test PubText',
]
TEST_TEXT_DIR = next((p for p in _pubtext_candidates if p.exists()), _pubtext_candidates[0])
print(f'PubText: {TEST_TEXT_DIR} — exists: {TEST_TEXT_DIR.exists()}')

PubText: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText — exists: True


## 1. Biological entity ontologies

These are the canonical values. Every extraction maps to one of these.
The hierarchy is encoded in the graph node types:
- `ORGANISM` — root node, NCBI taxon format
- `TISSUE` — UBERON format, child of ORGANISM
- `CELL_LINE` — specific line name, child of TISSUE (or ORGANISM)
- `CELL_TYPE` — primary cell type, child of TISSUE
- `DISEASE` — disease state, requires clinical context signal
- `PROTOCOL` — instrument, enzyme, label, modifications (independent)

In [2]:
ORGANISM_ONT = {
    'homo sapiens': '9606 (Homo sapiens)', 'human': '9606 (Homo sapiens)',
    'humans': '9606 (Homo sapiens)', 'patient': '9606 (Homo sapiens)',
    'mus musculus': '10090 (Mus musculus)', 'mouse': '10090 (Mus musculus)',
    'mice': '10090 (Mus musculus)', 'murine': '10090 (Mus musculus)',
    'rattus norvegicus': '10116 (Rattus norvegicus)', 'rat': '10116 (Rattus norvegicus)',
    'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
    'yeast': '4932 (Saccharomyces cerevisiae)',
    'escherichia coli': '562 (Escherichia coli)', 'e. coli': '562 (Escherichia coli)',
    'drosophila melanogaster': '7227 (Drosophila melanogaster)',
    'danio rerio': '7955 (Danio rerio)', 'zebrafish': '7955 (Danio rerio)',
    'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
    'sus scrofa': '9823 (Sus scrofa)', 'pig': '9823 (Sus scrofa)',
    'bos taurus': '9913 (Bos taurus)', 'bovine': '9913 (Bos taurus)',
    'gallus gallus': '9031 (Gallus gallus)', 'chicken': '9031 (Gallus gallus)',
    'caenorhabditis elegans': '6239 (Caenorhabditis elegans)',
    'xenopus laevis': '8355 (Xenopus laevis)',
    'macaca mulatta': '9544 (Macaca mulatta)',
    'oryctolagus cuniculus': '9986 (Oryctolagus cuniculus)', 'rabbit': '9986 (Oryctolagus cuniculus)',
}

TISSUE_ONT = {
    # Biofluids — longest match first
    'blood plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'blood serum': 'NT=blood serum;AC=UBERON:0001977',
    'whole blood': 'NT=blood;AC=UBERON:0000178',
    'peripheral blood': 'NT=blood;AC=UBERON:0000178',
    'cerebrospinal fluid': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'bronchoalveolar lavage': 'NT=bronchoalveolar lavage fluid;AC=UBERON:0000177',
    'synovial fluid': 'NT=synovial fluid;AC=UBERON:0001090',
    'plasma': 'NT=blood plasma;AC=UBERON:0001969',
    'serum': 'NT=blood serum;AC=UBERON:0001977',
    'blood': 'NT=blood;AC=UBERON:0000178',
    'urine': 'NT=urine;AC=UBERON:0001088',
    'saliva': 'NT=saliva;AC=UBERON:0001836',
    'csf': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
    # Brain regions
    'prefrontal cortex': 'NT=prefrontal cortex;AC=UBERON:0000451',
    'frontal cortex': 'NT=frontal cortex;AC=UBERON:0001870',
    'temporal cortex': 'NT=temporal lobe;AC=UBERON:0001871',
    'cerebral cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    'hippocampus': 'NT=hippocampal formation;AC=UBERON:0002421',
    'cerebellum': 'NT=cerebellum;AC=UBERON:0002037',
    'striatum': 'NT=striatum;AC=UBERON:0002435',
    'substantia nigra': 'NT=substantia nigra;AC=UBERON:0002038',
    'brain': 'NT=brain;AC=UBERON:0000955',
    'cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
    # Organs
    'bone marrow': 'NT=bone marrow;AC=UBERON:0002371',
    'adipose tissue': 'NT=adipose tissue;AC=UBERON:0001013',
    'skeletal muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'lymph node': 'NT=lymph node;AC=UBERON:0000029',
    'prostate gland': 'NT=prostate gland;AC=UBERON:0002367',
    'small intestine': 'NT=small intestine;AC=UBERON:0002108',
    'large intestine': 'NT=large intestine;AC=UBERON:0000059',
    'liver': 'NT=liver;AC=UBERON:0002107',
    'lung': 'NT=lung;AC=UBERON:0002048',
    'heart': 'NT=heart;AC=UBERON:0000948',
    'kidney': 'NT=kidney;AC=UBERON:0002113',
    'pancreas': 'NT=pancreas;AC=UBERON:0001264',
    'colon': 'NT=colon;AC=UBERON:0001155',
    'prostate': 'NT=prostate gland;AC=UBERON:0002367',
    'breast': 'NT=breast;AC=UBERON:0000310',
    'ovary': 'NT=ovary;AC=UBERON:0000992',
    'spleen': 'NT=spleen;AC=UBERON:0002106',
    'thymus': 'NT=thymus;AC=UBERON:0002370',
    'adipose': 'NT=adipose tissue;AC=UBERON:0001013',
    'muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
    'skin': 'NT=skin of body;AC=UBERON:0002097',
    'testis': 'NT=testis;AC=UBERON:0000473',
    'retina': 'NT=retina;AC=UBERON:0000966',
    'stomach': 'NT=stomach;AC=UBERON:0000945',
    'thyroid': 'NT=thyroid gland;AC=UBERON:0002046',
    'uterus': 'NT=uterus;AC=UBERON:0000995',
    'placenta': 'NT=placenta;AC=UBERON:0001987',
    # Cell populations
    'pbmc': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'peripheral blood mononuclear': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'platelet': 'NT=platelet;AC=CL:0000233',
    # Vesicles
    'extracellular vesicle': 'NT=extracellular vesicle;AC=GO:0061695',
    'exosome': 'NT=extracellular vesicle;AC=GO:0061695',
}

# Cell lines — these are DERIVED from tissue, never the organism itself
CELL_LINES = [
    'HEK293T','HEK293','HEK-293','HeLa','U2OS','MCF7','MCF-7','A549','Jurkat',
    'K562','HCT116','HepG2','CHO','PC3','LNCaP','THP-1','SH-SY5Y','Caco-2',
    'NIH3T3','RAW264.7','U87','U251','T47D','MDA-MB-231','MDA-MB-468','PANC-1',
    'MiaPaCa-2','AsPC-1','OVCAR-3','SKOV3','HL-60','HUVEC','B16','C2C12',
    '3T3-L1','U937','iPSC','DLD-1','RKO','Huh7','PC-9','H1975','A375',
    'SKBR3','BT474','ZR-75-1','HCC1954','IMR90','293T','PC12','A375',
    'Vero','BV2','MEF','Ramos','Daudi','U266','IM-9',
]

# Cell line to organism mapping (for validation)
CL_ORGANISM = {
    'HEK293T': '9606 (Homo sapiens)', 'HEK293': '9606 (Homo sapiens)',
    'HeLa': '9606 (Homo sapiens)', 'U2OS': '9606 (Homo sapiens)',
    'MCF7': '9606 (Homo sapiens)', 'A549': '9606 (Homo sapiens)',
    'Jurkat': '9606 (Homo sapiens)', 'K562': '9606 (Homo sapiens)',
    'HCT116': '9606 (Homo sapiens)', 'HepG2': '9606 (Homo sapiens)',
    'PC3': '9606 (Homo sapiens)', 'LNCaP': '9606 (Homo sapiens)',
    'THP-1': '9606 (Homo sapiens)', 'SH-SY5Y': '9606 (Homo sapiens)',
    'NIH3T3': '10090 (Mus musculus)', 'RAW264.7': '10090 (Mus musculus)',
    'B16': '10090 (Mus musculus)', 'C2C12': '10090 (Mus musculus)',
    '3T3-L1': '10090 (Mus musculus)', 'BV2': '10090 (Mus musculus)',
    'MEF': '10090 (Mus musculus)',
    'CHO': '9606 (Cricetulus griseus)',
    'Vero': '9606 (Chlorocebus sabaeus)',
}

# Cell line to tissue of origin (for MaterialType inference)
CL_TISSUE = {
    'HEK293T': 'kidney', 'HEK293': 'kidney', 'A549': 'lung',
    'HeLa': 'cervix', 'MCF7': 'breast', 'T47D': 'breast',
    'HCT116': 'colon', 'HepG2': 'liver', 'Huh7': 'liver',
    'Jurkat': 'blood', 'K562': 'blood', 'THP-1': 'blood',
    'PC3': 'prostate', 'LNCaP': 'prostate',
    'SH-SY5Y': 'brain', 'U87': 'brain', 'U251': 'brain',
    'PANC-1': 'pancreas', 'MiaPaCa-2': 'pancreas',
}

# Primary cell types (NOT cell lines — from living tissue)
CELL_TYPE_PATTERNS = [
    (re.compile(r'\b(neurons?|neuronal\s+cells?)\b', re.I), 'neurons'),
    (re.compile(r'\b(astrocytes?)\b', re.I), 'astrocytes'),
    (re.compile(r'\b(microglia)\b', re.I), 'microglia'),
    (re.compile(r'\b(macrophages?)\b', re.I), 'macrophages'),
    (re.compile(r'\b(fibroblasts?)\b', re.I), 'fibroblasts'),
    (re.compile(r'\b(t[\s\-]cells?|cd4\+|cd8\+)\b', re.I), 'T cells'),
    (re.compile(r'\b(b[\s\-]cells?)\b', re.I), 'B cells'),
    (re.compile(r'\b(pbmc|peripheral\s+blood\s+mononuclear)\b', re.I), 'PBMC'),
    (re.compile(r'\b(hepatocytes?)\b', re.I), 'hepatocytes'),
    (re.compile(r'\b(monocytes?)\b', re.I), 'monocytes'),
    (re.compile(r'\b(platelets?)\b', re.I), 'platelets'),
    (re.compile(r'\b(nk\s+cells?|natural\s+killer)\b', re.I), 'NK cells'),
    (re.compile(r'\b(dendritic\s+cells?)\b', re.I), 'dendritic cells'),
    (re.compile(r'\b(neutrophils?)\b', re.I), 'neutrophils'),
    (re.compile(r'\b(cardiomyocytes?)\b', re.I), 'cardiomyocytes'),
    (re.compile(r'\b(adipocytes?)\b', re.I), 'adipocytes'),
]

print('Ontologies loaded.')
print(f'  Organisms : {len(ORGANISM_ONT)}')
print(f'  Tissues   : {len(TISSUE_ONT)}')
print(f'  Cell lines: {len(CELL_LINES)}')

Ontologies loaded.
  Organisms : 29
  Tissues   : 55
  Cell lines: 58


## 2. Biological Relationship Graph

For each PXD we build a directed graph:
- Nodes: biological entities with type and canonical value
- Edges: `ORGANISM → derives → TISSUE`, `TISSUE → derives → CELL_LINE`

The graph enforces biological consistency:
- A cell line node makes the tissue node optional (cell lines don't need tissue)
- A cell line's organism must match the root organism node
- Disease nodes only added when clinical context is present
- DevelopmentalStage only added when it's consistent with MaterialType

In [3]:
def build_bio_graph(text, pride_data):
    """Build a biological relationship graph for a PXD from paper text + PRIDE API data.
    Returns a NetworkX DiGraph with typed nodes.
    """
    G = nx.DiGraph()
    text_low = text.lower() if text else ''

    # ── Step 1: Find ORGANISM (root node) ────────────────────────────────
    organisms = []

    # From PRIDE API (most reliable)
    for v in pride_data.get('Characteristics[Organism]', []):
        if v and v not in organisms:
            organisms.append(v)

    # From text (if PRIDE didn't provide)
    if not organisms:
        for key, norm in sorted(ORGANISM_ONT.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(r'\b' + re.escape(key) + r'\b', text_low):
                if norm not in organisms:
                    organisms.append(norm)

    for org in organisms:
        G.add_node(org, type='ORGANISM', col='Characteristics[Organism]', value=org)

    root_org = organisms[0] if organisms else None

    # ── Step 2: Find CELL LINES (before tissue — cell lines take priority) ─
    cell_lines_found = []

    # From PRIDE API source name / description
    for v in pride_data.get('Characteristics[CellLine]', []):
        if v and v not in cell_lines_found:
            cell_lines_found.append(v)

    # From text
    for cl in CELL_LINES:
        if re.search(r'\b' + re.escape(cl.lower()) + r'\b', text_low):
            if cl not in cell_lines_found:
                cell_lines_found.append(cl)

    for cl in cell_lines_found:
        G.add_node(cl, type='CELL_LINE', col='Characteristics[CellLine]', value=cl)
        # Edge: organism → cell_line
        cl_org = CL_ORGANISM.get(cl, root_org)
        if cl_org:
            if cl_org not in G:
                G.add_node(cl_org, type='ORGANISM', col='Characteristics[Organism]', value=cl_org)
            G.add_edge(cl_org, cl, relation='derives')

    # ── Step 3: Find TISSUE (only if no cell line, OR tissue is the source) ─
    # Key insight: if we have a cell line, the tissue is the cell line's tissue of origin
    # NOT the primary sample tissue
    tissues_found = []

    if not cell_lines_found:  # tissue is the sample source
        # From PRIDE API
        for v in pride_data.get('Characteristics[OrganismPart]', []):
            if v and v.startswith('NT=') and v not in tissues_found:
                tissues_found.append(v)

        # From text (longest match first)
        for tissue_key in sorted(TISSUE_ONT.keys(), key=len, reverse=True):
            if re.search(r'\b' + re.escape(tissue_key) + r'\b', text_low):
                norm = TISSUE_ONT[tissue_key]
                if norm not in tissues_found:
                    tissues_found.append(norm)

        for tissue in tissues_found:
            G.add_node(tissue, type='TISSUE', col='Characteristics[OrganismPart]', value=tissue)
            if root_org:
                G.add_edge(root_org, tissue, relation='source')

    else:
        # Cell line study — tissue is the derivation context
        # Only add tissue from PRIDE API, not from text (text tissue likely describes
        # where the cell line came from originally, not the experiment tissue)
        for v in pride_data.get('Characteristics[OrganismPart]', []):
            if v and v.startswith('NT='):
                G.add_node(v, type='TISSUE_ORIGIN', col='Characteristics[OrganismPart]', value=v)

    # ── Step 4: Find CELL TYPES (primary cells, not lines) ─────────────────
    # Only add if no cell lines found (otherwise ambiguous)
    if not cell_lines_found:
        cell_types_found = []
        for pat, val in CELL_TYPE_PATTERNS:
            if pat.search(text):
                cell_types_found.append(val)
        for ct in cell_types_found:
            G.add_node(ct, type='CELL_TYPE', col='Characteristics[CellType]', value=ct)
            # Connect to most specific tissue if available
            if tissues_found:
                G.add_edge(tissues_found[0], ct, relation='contains')
            elif root_org:
                G.add_edge(root_org, ct, relation='derives')

    # ── Step 5: Find DISEASE (requires clinical context) ──────────────────
    _CLINICAL = re.compile(
        r'\b(patient|cohort|biopsy|tumor|tumour|cancer|carcinoma|malignant|'
        r'diagnosed|clinical|disease|healthy\s+(?:control|donor)|specimen|'
        r'hospital|surgical|resection)\b', re.I)

    diseases_found = []
    if _CLINICAL.search(text):
        DISEASE_PATTERNS = [
            (re.compile(r'\b(alzheimer[\s\']?s?\s+disease)\b', re.I), 'Alzheimer disease'),
            (re.compile(r'\b(parkinson[\s\']?s?\s+disease)\b', re.I), 'Parkinson disease'),
            (re.compile(r'\b(type\s+2\s+diabetes(?:\s+mellitus)?|t2dm?)\b', re.I), 'type 2 diabetes mellitus'),
            (re.compile(r'\b(type\s+1\s+diabetes(?:\s+mellitus)?|t1dm?)\b', re.I), 'type 1 diabetes mellitus'),
            (re.compile(r'\b(breast\s+(?:cancer|carcinoma))\b', re.I), 'breast carcinoma'),
            (re.compile(r'\b(colorectal\s+(?:cancer|carcinoma)|colon\s+cancer)\b', re.I), 'colorectal carcinoma'),
            (re.compile(r'\b(non[\s\-]small[\s\-]cell\s+lung|nsclc)\b', re.I), 'non-small cell lung carcinoma'),
            (re.compile(r'\b(lung\s+(?:cancer|carcinoma|adenocarcinoma))\b', re.I), 'lung carcinoma'),
            (re.compile(r'\b(glioblastoma|gbm)\b', re.I), 'glioblastoma'),
            (re.compile(r'\b(melanoma)\b', re.I), 'melanoma'),
            (re.compile(r'\b(prostate\s+(?:cancer|carcinoma))\b', re.I), 'prostate carcinoma'),
            (re.compile(r'\b(ovarian\s+(?:cancer|carcinoma))\b', re.I), 'ovarian carcinoma'),
            (re.compile(r'\b(hepatocellular\s+carcinoma|hcc)\b', re.I), 'hepatocellular carcinoma'),
            (re.compile(r'\b(pancreatic\s+(?:cancer|ductal\s+adenocarcinoma)|pdac)\b', re.I), 'pancreatic ductal adenocarcinoma'),
            (re.compile(r'\b(covid[\s\-]?19|sars[\s\-]?cov[\s\-]?2)\b', re.I), 'COVID-19'),
            (re.compile(r'\b(multiple\s+myeloma)\b', re.I), 'multiple myeloma'),
            (re.compile(r'\b(acute\s+myeloid\s+leukemia|aml)\b', re.I), 'acute myeloid leukemia'),
            (re.compile(r'\b(osteoarthritis)\b', re.I), 'osteoarthritis'),
            (re.compile(r'\b(rheumatoid\s+arthritis)\b', re.I), 'rheumatoid arthritis'),
            (re.compile(r'\b(amyotrophic\s+lateral\s+sclerosis|als)\b', re.I), 'amyotrophic lateral sclerosis'),
            (re.compile(r'\b(healthy\s+(?:controls?|donors?|volunteers?|individuals?))\b', re.I), 'normal'),
        ]
        # Also from PRIDE
        for v in pride_data.get('Characteristics[Disease]', []):
            if v and v not in diseases_found: diseases_found.append(v)
        for pat, val in DISEASE_PATTERNS:
            if pat.search(text) and val not in diseases_found:
                diseases_found.append(val)

    for dis in diseases_found:
        G.add_node(dis, type='DISEASE', col='Characteristics[Disease]', value=dis)
        # Disease attaches to tissue or organism
        anchor = tissues_found[0] if tissues_found else root_org
        if anchor:
            G.add_edge(anchor, dis, relation='condition')

    return G


def graph_to_sdrf(G):
    """Read the biological graph and produce validated SDRF column assignments.
    Returns dict: col → list of values
    """
    out = defaultdict(list)

    cell_lines = [n for n,d in G.nodes(data=True) if d.get('type')=='CELL_LINE']
    cell_types  = [n for n,d in G.nodes(data=True) if d.get('type')=='CELL_TYPE']
    tissues     = [n for n,d in G.nodes(data=True) if d.get('type')=='TISSUE']
    organisms   = [n for n,d in G.nodes(data=True) if d.get('type')=='ORGANISM']
    diseases    = [n for n,d in G.nodes(data=True) if d.get('type')=='DISEASE']

    # Organism
    for org in organisms:
        out['Characteristics[Organism]'].append(G.nodes[org]['value'])

    # OrganismPart — only from tissue nodes (not cell line tissue origin)
    for tissue in tissues:
        out['Characteristics[OrganismPart]'].append(G.nodes[tissue]['value'])

    # CellLine — validated: must be consistent with organism
    root_org_val = organisms[0] if organisms else None
    for cl in cell_lines:
        cl_org = CL_ORGANISM.get(cl)
        # Validation: cell line organism must match root (or root unknown)
        if cl_org is None or root_org_val is None or cl_org == root_org_val:
            out['Characteristics[CellLine]'].append(cl)
        else:
            # Mismatch — could be a xenograft or contamination, skip
            pass

    # CellType — only if no cell lines (mutually exclusive in most studies)
    if not cell_lines:
        for ct in cell_types:
            out['Characteristics[CellType]'].append(G.nodes[ct]['value'])

    # Disease
    for dis in diseases:
        out['Characteristics[Disease]'].append(G.nodes[dis]['value'])

    # MaterialType — inferred from graph structure
    if cell_lines:
        out['Characteristics[MaterialType]'].append('cell line')
    elif cell_types:
        out['Characteristics[MaterialType]'].append('primary cells')
    elif tissues:
        # Is the tissue a biofluid?
        tissue_val = G.nodes[tissues[0]]['value']
        biofluid_terms = ['plasma','serum','urine','saliva','csf','blood']
        if any(t in tissue_val.lower() for t in biofluid_terms):
            out['Characteristics[MaterialType]'].append('biofluid')
        else:
            out['Characteristics[MaterialType]'].append('tissue')

    return dict(out)


print('Biological graph functions defined.')

# Quick test
test_text = ('HeLa cells derived from cervical cancer were used. '
             'Proteins from Homo sapiens were analyzed. '
             'Patients with breast carcinoma were recruited.')
test_pride = {}
G_test = build_bio_graph(test_text, test_pride)
print('\nTest graph nodes:')
for n, d in G_test.nodes(data=True):
    print(f'  [{d["type"]}] {n}')
print('\nTest graph edges:')
for u, v, d in G_test.edges(data=True):
    print(f'  {u} --{d["relation"]}--> {v}')
print('\nSDRF assignments:')
for col, vals in graph_to_sdrf(G_test).items():
    print(f'  {col}: {vals}')

Biological graph functions defined.

Test graph nodes:
  [ORGANISM] 9606 (Homo sapiens)
  [CELL_LINE] HeLa
  [DISEASE] breast carcinoma

Test graph edges:
  9606 (Homo sapiens) --derives--> HeLa
  9606 (Homo sapiens) --condition--> breast carcinoma

SDRF assignments:
  Characteristics[Organism]: ['9606 (Homo sapiens)']
  Characteristics[CellLine]: ['HeLa']
  Characteristics[Disease]: ['breast carcinoma']
  Characteristics[MaterialType]: ['cell line']


## 3. Load training data, PRIDE API, text, and protocol regex

In [4]:
# Training data
sample_sub  = pd.read_csv(SAMPLE_SUB)
id_cols     = ['ID','PXD','Raw Data File','Usage']
target_cols = [c for c in sample_sub.columns
               if c not in id_cols and 'Unnamed' not in c]
all_base    = set(re.sub(r'\.\d+$','',c) for c in target_cols)

def _find_col(col, df_cols):
    if col in df_cols: return col
    base = re.sub(r'\.\d+$','',col)
    if base in df_cols: return base
    m = re.match(r'(?:characteristics|comment|factor\s*value)\[(.+?)\]', base, re.I)
    if m and m.group(1) in df_cols: return m.group(1)
    return None

col_counters = {col: Counter() for col in target_cols}
train_pxd_sdrf = {}
train_files = []
if TRAIN_SDRF_DIR.exists():
    train_files = list(TRAIN_SDRF_DIR.glob('*.tsv')) + list(TRAIN_SDRF_DIR.glob('*.csv'))

for fp in train_files:
    sep = '\t' if fp.suffix == '.tsv' else ','
    try: df = pd.read_csv(fp, low_memory=False, sep=sep)
    except: continue
    pxd = fp.stem.replace('Harmonized_','').replace('_cleaned.sdrf','').split('.')[0]
    pxd_vals = {}
    for col in target_cols:
        mc = _find_col(col, set(df.columns))
        if mc:
            vals = df[mc].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','n/a','na',''])]
            col_counters[col].update(vals.tolist())
            uniq = list(vals.unique())
            if uniq: pxd_vals[col] = uniq
    train_pxd_sdrf[pxd] = pxd_vals

global_modes = {}
non_na_ratio = {}
n_train = max(len(train_files), 1)
for col in target_cols:
    total = sum(col_counters[col].values())
    global_modes[col] = col_counters[col].most_common(1)[0][0] if total > 0 else 'Not Applicable'
    non_na_ratio[col] = total / n_train if total > 0 else 0.0

NO_FALLBACK = {
    'Characteristics[SyntheticPeptide]','Characteristics[PooledSample]',
    'Characteristics[Bait]','Characteristics[TumorSize]','Characteristics[GrowthRate]',
    'Characteristics[SamplingTime]','Characteristics[Time]','Characteristics[Compound]',
    'Characteristics[ConcentrationOfCompound]','Characteristics[Treatment]',
    'Characteristics[DiseaseTreatment]','Characteristics[Depletion]',
    'Characteristics[CellPart]','Characteristics[Age]','Characteristics[BMI]',
    'Characteristics[AncestryCategory]','Characteristics[Disease]',
    'Characteristics[CellLine]','Characteristics[CellType]',
    'FactorValue[Bait]','FactorValue[CellPart]','FactorValue[Treatment]',
    'FactorValue[Disease]','FactorValue[Compound]',
    'FactorValue[ConcentrationOfCompound].1','FactorValue[GeneticModification]',
    'FactorValue[Temperature]','FactorValue[FractionIdentifier]',
}

print(f'Train SDRFs : {len(train_files)}')
print(f'Target cols : {len(target_cols)}')

Train SDRFs : 103
Target cols : 77


In [5]:
# PRIDE API
http_session = requests.Session()
http_session.headers.update({'User-Agent': 'SDRF-Graph/1.0'})

def fetch_pride(pxd):
    try:
        r = http_session.get(
            f'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{pxd}',
            timeout=PRIDE_TIMEOUT)
        if r.status_code != 200: return {}
        d = r.json()
        out = defaultdict(list)
        for o in d.get('organisms',[]):
            name = o.get('name','')
            if name:
                norm = next((v for k,v in ORGANISM_ONT.items()
                             if k in name.lower().strip()), None)
                if norm: out['Characteristics[Organism]'].append(norm)
        for op in (d.get('organisms_part') or d.get('tissues') or []):
            name = op.get('name',''); acc = op.get('accession','')
            if name and name.lower() not in ('not available','n/a',''):
                norm = next((v for k,v in sorted(TISSUE_ONT.items(),
                             key=lambda x: len(x[0]), reverse=True)
                             if k in name.lower()), None)
                if norm: out['Characteristics[OrganismPart]'].append(norm)
                elif acc: out['Characteristics[OrganismPart]'].append(f'NT={name};AC={acc}')
        for dis in d.get('diseases',[]):
            name = dis.get('name','')
            if name and name.lower() not in ('not available','n/a','none','normal',''):
                out['Characteristics[Disease]'].append(name)
        for inst in d.get('instruments',[]):
            name = inst.get('name',''); acc = inst.get('accession','')
            if name:
                # Normalize instrument
                INST = {'q exactive hf-x':'AC=MS:1003027;NT=Q Exactive HF-X',
                        'q exactive hf':'AC=MS:1002523;NT=Q Exactive HF',
                        'q exactive plus':'AC=MS:1002634;NT=Q Exactive Plus',
                        'q exactive':'AC=MS:1001911;NT=Q Exactive',
                        'orbitrap astral':'AC=MS:1003378;NT=Orbitrap Astral',
                        'orbitrap fusion lumos':'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
                        'fusion lumos':'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
                        'orbitrap fusion':'AC=MS:1002416;NT=Orbitrap Fusion',
                        'orbitrap eclipse':'AC=MS:1003029;NT=Orbitrap Eclipse',
                        'orbitrap exploris 480':'AC=MS:1003094;NT=Orbitrap Exploris 480',
                        'exploris 480':'AC=MS:1003094;NT=Orbitrap Exploris 480',
                        'ltq orbitrap velos':'AC=MS:1001742;NT=LTQ Orbitrap Velos',
                        'ltq orbitrap elite':'AC=MS:1001910;NT=LTQ Orbitrap Elite',
                        'ltq orbitrap xl':'AC=MS:1000556;NT=LTQ Orbitrap XL',
                        'ltq orbitrap':'AC=MS:1000449;NT=LTQ Orbitrap',
                        'timstof pro':'AC=MS:1003231;NT=timsTOF Pro',
                        'timstof':'AC=MS:1002817;NT=timsTOF',
                        'triple tof 6600':'AC=MS:1000931;NT=TripleTOF 6600',
                        'triple tof 5600':'AC=MS:1000931;NT=TripleTOF 5600',
                        'triple tof':'AC=MS:1000931;NT=TripleTOF 6600',
                        'synapt g2':'AC=MS:1002726;NT=Synapt G2-Si',
                        'velos pro':'AC=MS:1001909;NT=LTQ Velos Pro',
                        }
                n = name.lower().strip()
                norm = next((v for k,v in sorted(INST.items(),
                             key=lambda x: len(x[0]), reverse=True)
                             if k in n), None)
                if norm: out['Comment[Instrument]'].append(norm)
                elif acc: out['Comment[Instrument]'].append(f'AC={acc};NT={name}')
        for qm in d.get('quantification_methods',[]):
            name = qm.get('name','')
            if name:
                def fmt_label(n):
                    n = str(n).lower().strip()
                    if any(x in n for x in ['label free','label-free','lfq']):
                        return 'AC=MS:1002038;NT=label free sample'
                    if 'tmt' in n:
                        m = re.search(r'tmt[\s\-]?(\d+)', n)
                        p = m.group(1) if m else '6'
                        acc = {'2':'MS:1002456','6':'MS:1002453','10':'MS:1002454',
                               '11':'MS:1002454','16':'MS:1003998','18':'MS:1003999'}
                        return f'AC={acc.get(p,"MS:1002453")};NT=TMT{p}plex'
                    if 'silac' in n: return 'AC=MS:1002791;NT=SILAC'
                    if 'itraq' in n:
                        m = re.search(r'itraq[\s\-]?(\d+)', n)
                        p = m.group(1) if m else '4'
                        return f"AC={'MS:1001985' if p=='4' else 'MS:1002519'};NT=iTRAQ{p}plex"
                    return str(n)
                out['Characteristics[Label]'].append(fmt_label(name))
        return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}
    except Exception as e:
        print(f'  PRIDE {pxd}: {e}')
        return {}

print('PRIDE API ready.')

PRIDE API ready.


In [6]:
# Protocol regex (instrument-level, not biological)
_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'

SECTIONS = ['TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
            'EXPERIMENTAL','SAMPLE PREPARATION','MASS SPECTROMETRY',
            'LC-MS','LC-MS/MS','PROTEIN DIGESTION','DATA ACQUISITION',
            'DATA ANALYSIS','CELL CULTURE']
METHOD_KWS = ['method','material','protocol','digest','spectr',
              'chromat','prep','enrichment','culture','experimental']

def get_text(pub_dict):
    parts = []
    for key in ['TITLE','ABSTRACT'] + SECTIONS:
        v = pub_dict.get(key,'')
        if isinstance(v,list): v=' '.join(str(x) for x in v)
        if v.strip(): parts.append(v.strip())
    for key,v in pub_dict.items():
        if key.upper() in SECTIONS+['TITLE','ABSTRACT']: continue
        if any(kw in key.lower() for kw in METHOD_KWS):
            if isinstance(v,list): v=' '.join(str(x) for x in v)
            if v.strip(): parts.append(v.strip())
    return ' '.join(parts)


def fmt_label(n):
    n = str(n).lower().strip()
    if any(x in n for x in ['label free','label-free','lfq']):
        return 'AC=MS:1002038;NT=label free sample'
    if 'tmt' in n:
        m = re.search(r'tmt[\s\-]?(\d+)', n)
        p = m.group(1) if m else '6'
        acc = {'2':'MS:1002456','6':'MS:1002453','10':'MS:1002454',
               '11':'MS:1002454','16':'MS:1003998','18':'MS:1003999'}
        return f'AC={acc.get(p,"MS:1002453")};NT=TMT{p}plex'
    if 'itraq' in n:
        m = re.search(r'itraq[\s\-]?(\d+)', n)
        p = m.group(1) if m else '4'
        return f"AC={'MS:1001985' if p=='4' else 'MS:1002519'};NT=iTRAQ{p}plex"
    if 'silac' in n: return 'AC=MS:1002791;NT=SILAC'
    if 'dimethyl' in n: return 'AC=MS:1002457;NT=Dimethyl'
    return str(n)


def protocol_regex(pub_dict):
    """Extract protocol-level fields (instrument, enzyme, label, modifications).
    These are NOT part of the biological hierarchy."""
    text = get_text(pub_dict)
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]: out[col].append(val)

    INST_PATS = [
        (re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF[\s\-]?X)\b',re.I),'AC=MS:1003027;NT=Q Exactive HF-X'),
        (re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?HF)\b',re.I),'AC=MS:1002523;NT=Q Exactive HF'),
        (re.compile(r'\b(Q[\s\-]?Exactive[\s\-]?Plus)\b',re.I),'AC=MS:1002634;NT=Q Exactive Plus'),
        (re.compile(r'\b(Q[\s\-]?Exactive)\b',re.I),'AC=MS:1001911;NT=Q Exactive'),
        (re.compile(r'\b(Orbitrap\s+Astral)\b',re.I),'AC=MS:1003378;NT=Orbitrap Astral'),
        (re.compile(r'\b(Orbitrap\s+Fusion\s+Lumos)\b',re.I),'AC=MS:1002732;NT=Orbitrap Fusion Lumos'),
        (re.compile(r'\b(Orbitrap\s+Fusion)\b',re.I),'AC=MS:1002416;NT=Orbitrap Fusion'),
        (re.compile(r'\b(Orbitrap\s+Eclipse)\b',re.I),'AC=MS:1003029;NT=Orbitrap Eclipse'),
        (re.compile(r'\b(Orbitrap\s+Exploris\s+480|Exploris\s+480)\b',re.I),'AC=MS:1003094;NT=Orbitrap Exploris 480'),
        (re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Velos)\b',re.I),'AC=MS:1001742;NT=LTQ Orbitrap Velos'),
        (re.compile(r'\b(LTQ[\s\-]?Orbitrap\s+Elite)\b',re.I),'AC=MS:1001910;NT=LTQ Orbitrap Elite'),
        (re.compile(r'\b(LTQ[\s\-]?Orbitrap)\b',re.I),'AC=MS:1000449;NT=LTQ Orbitrap'),
        (re.compile(r'\b(timsTOF\s+Pro\s+2)\b',re.I),'AC=MS:1003474;NT=timsTOF Pro 2'),
        (re.compile(r'\b(timsTOF\s+Pro)\b',re.I),'AC=MS:1003231;NT=timsTOF Pro'),
        (re.compile(r'\b(timsTOF)\b',re.I),'AC=MS:1002817;NT=timsTOF'),
        (re.compile(r'\b(Triple[\s\-]?TOF\s+6600)\b',re.I),'AC=MS:1000931;NT=TripleTOF 6600'),
        (re.compile(r'\b(Triple[\s\-]?TOF\s+5600)\b',re.I),'AC=MS:1000931;NT=TripleTOF 5600'),
        (re.compile(r'\b(Triple[\s\-]?TOF)\b',re.I),'AC=MS:1000931;NT=TripleTOF 6600'),
    ]
    for pat, val in INST_PATS:
        if pat.search(text): add('Comment[Instrument]', val); break

    for pat, val in [
        (re.compile(_NEG+r'\b(trypsin(?:/lys[\s\-]?c)?)\b',re.I),'AC=MS:1001251;NT=Trypsin'),
        (re.compile(_NEG+r'\b(lys[\s\-]?c)\b',re.I),'AC=MS:1001255;NT=Lys-C'),
        (re.compile(_NEG+r'\b(glu[\s\-]?c)\b',re.I),'AC=MS:1001917;NT=Glu-C'),
        (re.compile(_NEG+r'\b(chymotrypsin)\b',re.I),'AC=MS:1001306;NT=Chymotrypsin'),
        (re.compile(_NEG+r'\b(asp[\s\-]?n)\b',re.I),'AC=MS:1001267;NT=Asp-N'),
        (re.compile(_NEG+r'\b(arg[\s\-]?c)\b',re.I),'AC=MS:1001303;NT=Arg-C'),
    ]:
        if pat.search(text): add('Characteristics[CleavageAgent]', val); break

    for pat, fn in [
        (re.compile(r'\b(tmt[\s\-]?(?:pro|18|16|11|10|6|2)?(?:plex)?)\b',re.I), fmt_label),
        (re.compile(r'\b(itraq[\s\-]?(?:4|8)?(?:plex)?)\b',re.I), fmt_label),
        (re.compile(r'\b(silac)\b',re.I), lambda _: 'AC=MS:1002791;NT=SILAC'),
        (re.compile(r'\b(label[\s\-]free|lfq)\b',re.I), lambda _: 'AC=MS:1002038;NT=label free sample'),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[Label]', fn(m.group(1))); break

    for pat, val in [
        (re.compile(_NEG+r'\b(dtt|dithiothreitol)\b',re.I),'AC=MS:1000578;NT=DTT'),
        (re.compile(_NEG+r'\b(tcep)\b',re.I),'AC=MS:1001135;NT=TCEP'),
    ]:
        if pat.search(text): add('Characteristics[ReductionReagent]', val); break

    for pat, val in [
        (re.compile(r'\b(iodoacetamide|iaa)\b',re.I),'AC=PRIDE:0000126;NT=Iodoacetamide'),
        (re.compile(r'\b(n[\s\-]?ethylmaleimide|nem)\b',re.I),'AC=PRIDE:0000459;NT=N-ethylmaleimide'),
        (re.compile(r'\b(chloroacetamide|caa)\b',re.I),'AC=PRIDE:0000126;NT=Chloroacetamide'),
    ]:
        if pat.search(text): add('Characteristics[AlkylationReagent]', val); break

    for pat, val in [
        (re.compile(r'\b(carbamidomethyl(?:ation)?)\b',re.I),'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed'),
        (re.compile(r'\b(oxidation)\b',re.I),'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable'),
        (re.compile(r'\b(phospho(?:rylation)?)\b',re.I),'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable'),
        (re.compile(r'\b(acetyl(?:ation)?)\b',re.I),'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable'),
        (re.compile(r'\b(ubiquitin(?:ation)?|di[\s\-]?glycine|gg[\s\-]?remnant)\b',re.I),'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable'),
        (re.compile(r'\b(methylation)\b',re.I),'NT=Methyl;AC=UNIMOD:34;TA=K,R;MT=Variable'),
        (re.compile(r'\b(deamidation|deamidated)\b',re.I),'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable'),
    ]:
        if pat.search(text): add('Characteristics[Modification]', val)

    for pat, val in [
        (re.compile(r'\b(hcd)\b',re.I),'AC=MS:1002481;NT=HCD'),
        (re.compile(r'\b(cid)\b',re.I),'AC=MS:1001880;NT=CID'),
        (re.compile(r'\b(etd)\b',re.I),'AC=MS:1001526;NT=ETD'),
    ]:
        if pat.search(text): add('Comment[FragmentationMethod]', val)

    for pat, val in [
        (re.compile(r'\b(dda|data[\s\-]dependent)\b',re.I),'AC=MS:1003215;NT=DDA'),
        (re.compile(r'\b(dia|data[\s\-]independent|swath)\b',re.I),'AC=MS:1003215;NT=DIA'),
        (re.compile(r'\b(prm|parallel\s+reaction\s+monitoring)\b',re.I),'AC=MS:1001501;NT=PRM'),
    ]:
        if pat.search(text): add('Comment[AcquisitionMethod]', val); break

    if re.search(r'\b(nano[\s\-]?esi|nesi)\b',text,re.I):
        add('Comment[IonizationType]','AC=MS:1000398;NT=nanoESI')
    elif re.search(r'\b(electrospray|esi)\b',text,re.I):
        add('Comment[IonizationType]','AC=MS:1000073;NT=ESI')

    for pat, val in [
        (re.compile(r'\b(orbitrap)\b',re.I),'AC=MS:1000484;NT=Orbitrap'),
        (re.compile(r'\b(ion\s*trap)\b',re.I),'AC=MS:1000264;NT=ion trap'),
        (re.compile(r'\b(tof)\b',re.I),'AC=MS:1000084;NT=TOF'),
    ]:
        if pat.search(text): add('Comment[MS2MassAnalyzer]', val); break

    for pat, val in [
        (re.compile(r'\b(sds[\s\-]?page)\b',re.I),'AC=PRIDE:0000672;NT=SDS-PAGE'),
        (re.compile(r'\b(scx|strong\s+cation\s+exchange)\b',re.I),'AC=PRIDE:0000228;NT=SCX'),
        (re.compile(r'\b(hprp|high[\s\-]?ph\s+(?:rp|reversed[\s\-]phase))\b',re.I),'AC=PRIDE:0000550;NT=High-pH Reversed-Phase'),
    ]:
        if pat.search(text): add('Comment[FractionationMethod]', val); break

    for pat, val in [
        (re.compile(r'\b(tio2?|titanium\s+dioxide)\b',re.I),'AC=MS:1002088;NT=TiO2'),
        (re.compile(r'\b(imac|immobilized\s+metal\s+affinity)\b',re.I),'AC=MS:1001923;NT=IMAC'),
        (re.compile(r'\b(immunoprecipitation|ip[\s\-]ms)\b',re.I),'AC=MS:1002090;NT=Immunoprecipitation'),
    ]:
        if pat.search(text): add('Comment[EnrichmentMethod]', val); break

    if re.search(r'\b(nano[\s\-]?lc)\b',text,re.I):
        add('Comment[Separation]','AC=PRIDE:0000565;NT=nanoLC')
    elif re.search(r'\b(uplc|uhplc)\b',text,re.I):
        add('Comment[Separation]','UPLC')

    if re.search(r'\b(male\s+and\s+female|both\s+sexes)\b',text,re.I):
        add('Characteristics[Sex]','male and female')
    elif re.search(r'\b(male\s+(?:donors?|subjects?|patients?|mice|rats?))\b',text,re.I):
        add('Characteristics[Sex]','male')
    elif re.search(r'\b(female\s+(?:donors?|subjects?|patients?|mice|rats?))\b',text,re.I):
        add('Characteristics[Sex]','female')

    for pat, val in [
        (re.compile(r'\b(C57BL/6J?)\b'),'C57BL/6J'),
        (re.compile(r'\b(BALB/c)\b'),'BALB/c'),
        (re.compile(r'\b(Sprague[\s\-]Dawley)\b',re.I),'Sprague-Dawley'),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[Strain]', val); break

    for pat, val in [
        (re.compile(r'\b(wild[\s\-]?type|wt(?:\s+cells?|\s+mice)?)\b',re.I),'wild-type'),
        (re.compile(r'\b(knockout|knock[\s\-]out|ko(?:\s+cells?|\s+mice)?)\b',re.I),'knockout'),
        (re.compile(r'\b(transgenic)\b',re.I),'transgenic'),
    ]:
        if pat.search(text): add('Characteristics[Genotype]', val); break

    # Numeric
    for pat in [
        re.compile(r'(\d+)[\s\-]min(?:ute)?\s+(?:gradient|linear\s+gradient)\b',re.I),
        re.compile(r'gradient\s+(?:of\s+)?(\d+)[\s\-]?min\b',re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[GradientTime]', f'{m.group(1)} min'); break

    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL|µl|µL|ul|uL)\s*/\s*min', text)
    if m:
        unit = 'nL' if m.group(2).lower()=='nl' else 'µL'
        add('Comment[FlowRateChromatogram]', f'{m.group(1)} {unit}/min')

    for pat in [
        re.compile(r'(?:precursor|ms1)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da)',re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*ppm\s+(?:for\s+)?(?:precursor|ms1)',re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex>=2 else 'ppm'
            add('Comment[PrecursorMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:fragment|ms2)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da|mda)',re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*(da|mda)\s+(?:for\s+)?(?:fragment|ms2)',re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex>=2 else 'Da'
            add('Comment[FragmentMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:up\s+to\s+)(\d)\s+missed\s+cleavages?',re.I),
        re.compile(r'missed\s+cleavages?\s*[=:≤]\s*(\d)',re.I),
    ]:
        m = pat.search(text)
        if m: add('Comment[NumberOfMissedCleavages]', m.group(1)); break

    for pat in [
        re.compile(r'(\d+)\s+(?:independent\s+)?biological\s+replicates?',re.I),
        re.compile(r'performed\s+in\s+(triplicate|duplicate)\b',re.I),
    ]:
        m = pat.search(text)
        if m:
            wm = {'triplicate':'3','duplicate':'2'}
            val = wm.get(m.group(1).lower() if m.lastindex else '', m.group(1) if m.lastindex else '3')
            add('Characteristics[NumberOfBiologicalReplicates]', val); break

    for pat in [
        re.compile(r'cohort\s+of\s+(\d+)\s+(?:patients?|subjects?)',re.I),
        re.compile(r'total\s+of\s+(\d+)\s+samples?',re.I),
    ]:
        m = pat.search(text)
        if m: add('Characteristics[NumberOfSamples]', m.group(1)); break

    m = re.search(r'\b(?:stage\s+)([IViv]+)\b',text,re.I)
    if m: add('Characteristics[TumorStage]', f'Stage {m.group(1).upper()}')

    for col in ['Characteristics[Modification]']:
        if col in out: out[col] = out[col][:4]

    return dict(out)

print('Protocol regex defined.')

Protocol regex defined.


## 4. Load test papers

In [7]:
test_docs = {}
pxd_to_raws = {}
for _, row in sample_sub.iterrows():
    pxd_to_raws.setdefault(row['PXD'],[]).append(row['Raw Data File'])

if TEST_TEXT_DIR.exists():
    for fp in sorted(TEST_TEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: test_docs[pxd] = d
        except: pass

print(f'Test papers : {len(test_docs)}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

Test papers : 16
  PubText: 0 chars
  PXD004010: 11,237 chars
  PXD016436: 7,509 chars
  PXD019519: 45,042 chars
  PXD025663: 20,717 chars
  PXD040582: 19,714 chars
  PXD050621: 14,085 chars
  PXD061009: 34,629 chars
  PXD061090: 19,600 chars
  PXD061136: 15,870 chars
  PXD061195: 9,808 chars
  PXD061285: 34,103 chars
  PXD062014: 29,451 chars
  PXD062469: 16,510 chars
  PXD062877: 35,268 chars
  PXD064564: 19,510 chars


## 5. Main pipeline

For each PXD:
1. Fetch PRIDE API data
2. Build biological relationship graph from text + PRIDE
3. Read validated SDRF assignments from graph
4. Add protocol fields from regex (these don't need hierarchy validation)
5. Apply conservative fallback for high-frequency stable columns

In [9]:
def parse_fraction(rf):
    for p in [r'[_\-\.](f|fr|frac(?:tion)?)[_\-\.\s]?(\d{1,3})(?=[_\-\.]|$)',
              r'fraction(\d{1,3})', r'(\d{1,3})of\d+']:
        m = re.search(p, str(rf), re.I)
        if m:
            n = m.group(2) if m.lastindex and m.lastindex>=2 else m.group(1)
            if n and n.isdigit() and 1<=int(n)<=200: return str(int(n))
    return None

def parse_biol_rep(rf):
    for p in [r'[_\-]biolrep[_\-]?(\d+)', r'[_\-]br(\d+)[_\-\.]',
              r'[_\-]rep(\d+)[_\-\.]', r'[_\-]r(\d{1,2})[_\-\.]']:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1<=int(m.group(1))<=50:
            return str(int(m.group(1)))
    return None

def parse_label_from_fn(rf):
    ru = str(rf).upper()
    m = re.search(r'TMT(PRO|18|16|11|10|6|2)', ru)
    if m:
        pmap={'PRO':'16','18':'18','16':'16','11':'11','10':'10','6':'6','2':'2'}
        amap={'18':'MS:1003999','16':'MS:1003998','11':'MS:1002454',
              '10':'MS:1002454','6':'MS:1002453','2':'MS:1002456'}
        p = pmap.get(m.group(1),'6')
        return f'AC={amap[p]};NT=TMT{p}plex'
    if 'TMT' in ru: return 'AC=MS:1002453;NT=TMT6plex'
    if re.search(r'SILAC|_H_|_HVY|_L_', ru): return 'AC=MS:1002791;NT=SILAC'
    if re.search(r'LFQ|LABELFREE|_LF_', ru): return 'AC=MS:1002038;NT=label free sample'
    return None


final_sub = pd.read_csv(SAMPLE_SUB, dtype=str).copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

for pxd, pxd_df in tqdm(final_sub.groupby('PXD'), desc='PXDs'):
    idx       = pxd_df.index
    raw_files = pxd_to_raws[pxd]
    pub_dict  = test_docs.get(pxd, {})
    text      = get_text(pub_dict) if pub_dict else ''

    # Layer 0: training overlap (ground truth from training set)
    pxd_vals = defaultdict(list)
    def pxd_add(col, val):
        if not val: return
        v = str(val).strip()
        if v.lower() in ('not applicable','na','n/a','','null','none'): return
        if v not in pxd_vals[col]: pxd_vals[col].append(v)

    if pxd in train_pxd_sdrf:
        for col, vals in train_pxd_sdrf[pxd].items():
            for v in (vals or []): pxd_add(col, v)

    # Layer 1: PRIDE API
    pride_data = fetch_pride(pxd)
    time.sleep(0.3)

    # Layer 2: Build biological relationship graph
    G = build_bio_graph(text, pride_data)
    bio_assignments = graph_to_sdrf(G)

    # Write graph-validated biological assignments
    for col, vals in bio_assignments.items():
        for v in vals: pxd_add(col, v)

    # Also write PRIDE protocol fields not in graph
    for col in ['Comment[Instrument]','Characteristics[Label]']:
        for v in pride_data.get(col, []):
            pxd_add(col, v)

    # Layer 3: Protocol regex (instrument, enzyme, mods — no hierarchy needed)
    if pub_dict:
        for col, vals in protocol_regex(pub_dict).items():
            if isinstance(vals, list):
                for v in vals: pxd_add(col, v)
            else:
                pxd_add(col, vals)

    # Layer 4: Conservative fallback
    filled_bases = set(re.sub(r'\.\d+$','',c) for c in pxd_vals)
    for col in target_cols:
        base = re.sub(r'\.\d+$','',col)
        if base in filled_bases: continue
        if col in NO_FALLBACK: continue
        total = sum(col_counters[col].values())
        if total > 0:
            top_val, top_count = col_counters[col].most_common(1)[0]
            if top_count/total > 0.80 and non_na_ratio.get(col,0.0) > 0.80:
                pxd_add(col, top_val)

    # Modification slots
    mods = list(dict.fromkeys(pxd_vals.pop('Characteristics[Modification]',[])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i==0 else f'Characteristics[Modification].{i}'
        pxd_vals[slot] = [mod]

    # Write per-file
    for i, (row_idx, raw_file) in enumerate(zip(idx, raw_files)):
        fraction = parse_fraction(raw_file)
        biol_rep = parse_biol_rep(raw_file)
        fn_label = parse_label_from_fn(raw_file)

        if fraction: final_sub.at[row_idx,'Comment[FractionIdentifier]'] = fraction
        if biol_rep: final_sub.at[row_idx,'Characteristics[BiologicalReplicate]'] = biol_rep
        if fn_label and final_sub.at[row_idx,'Characteristics[Label]']=='Not Applicable':
            final_sub.at[row_idx,'Characteristics[Label]'] = fn_label

        for col in target_cols:
            if final_sub.at[row_idx,col] != 'Not Applicable': continue
            base = re.sub(r'\.\d+$','',col)
            vals = pxd_vals.get(col) or pxd_vals.get(base) or []
            vals = [v for v in vals if str(v).strip().lower() not in ('not applicable','')]
            if vals: final_sub.at[row_idx,col] = vals[i % len(vals)]

# Cleanup
final_sub = final_sub.fillna('Not Applicable')
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['nan','None','[]','','null','not available','TextSpan','not applicable'])
    final_sub.loc[mask,col] = 'Not Applicable'

# FractionIdentifier artifact cleanup
for pxd, grp in final_sub.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs)==1 and str(fracs[0]).strip() in ('1','Not Applicable'):
        final_sub.loc[grp.index,'Comment[FractionIdentifier]'] = 'Not Applicable'

final_sub.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')

PXDs: 100%|██████████| 15/15 [00:16<00:00,  1.07s/it]


Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_graph.csv
Shape : (1659, 81)


In [10]:
# Fill rate + spot check
label_cols = [c for c in final_sub.columns
              if c not in ('ID','PXD','Raw Data File','Usage')]
rows = [(c,(final_sub[c]!='Not Applicable').sum()) for c in label_cols]
rows.sort(key=lambda x:-x[1])
print(f'{"Column":<55} {"Filled":>7} {"Pct":>6}')
print('-'*72)
for col,n in rows:
    if n>0: print(f'{col:<55} {n:>7} {n/len(final_sub)*100:>5.1f}%')
print(f'\nTotal filled: {sum(1 for _,n in rows if n>0)} / {len(rows)}')

print('\n=== Spot check ===')
for col in ['Characteristics[SyntheticPeptide]','Characteristics[Bait]',
            'Characteristics[Organism]','Characteristics[OrganismPart]',
            'Characteristics[CellLine]','Characteristics[Disease]',
            'Comment[MS2MassAnalyzer]']:
    if col in final_sub.columns:
        vc = final_sub[col].value_counts().head(3)
        na = (final_sub[col]=='Not Applicable').sum()
        print(f'\n{col} (NA={na}):')
        for v,n in vc.items():
            print(f'  {str(v)[:70]}: {n}')

print('\n=== Graph output per PXD ===')
for pxd in sorted(final_sub['PXD'].unique()):
    grp = final_sub[final_sub['PXD']==pxd]
    org = grp['Characteristics[Organism]'].value_counts().index[0]
    part = grp['Characteristics[OrganismPart]'].value_counts().index[0]
    cl = grp['Characteristics[CellLine]'].value_counts().index[0]
    dis = grp['Characteristics[Disease]'].value_counts().index[0]
    print(f'{pxd}: org={str(org)[:25]} | part={str(part)[:30]} | cl={cl[:15]} | dis={dis[:20]}')

Column                                                   Filled    Pct
------------------------------------------------------------------------
Characteristics[BiologicalReplicate]                       1659 100.0%
Characteristics[Organism]                                  1659 100.0%
Comment[Instrument]                                        1659 100.0%
Comment[MS2MassAnalyzer]                                   1659 100.0%
Characteristics[MaterialType]                              1632  98.4%
Characteristics[CleavageAgent]                             1610  97.0%
Characteristics[Modification]                              1603  96.6%
Characteristics[Modification].1                            1603  96.6%
Characteristics[Modification].2                            1603  96.6%
Characteristics[Modification].3                            1603  96.6%
Characteristics[Modification].4                            1603  96.6%
Characteristics[Modification].5                            1603  96.6%
Char